# LeetCode #1204: Design Hit Counter

https://leetcode.com/problems/design-hit-counter/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(1)$ hit, $O(n)$ getHits | $O(n)$ |
| **Optimal: Circular Buffer (300 slots) ★** | $O(1)$ hit, $O(300) = O(1)$ getHits | $O(1)$ |

---

## Understanding the Methods

### Brute Force
Store every hit timestamp in a queue. On `getHits(t)`, pop all timestamps older than $t - 300$, then return the queue size. $O(n)$ per `getHits` when many hits accumulate.

### Optimal: Circular Buffer (300 slots) ★
Maintain exactly 300 slots indexed by `timestamp % 300`. Each slot stores the last timestamp that wrote to it plus its hit count. On `hit(t)`, reset the slot if it belongs to an older second; on `getHits(t)`, sum only the slots within the last 300 seconds.

**Why this is better than Brute Force:** Caps space at exactly 300 pairs regardless of total hits; both operations are bounded by 300 iterations — effectively $O(1)$.

**Constraints:**
* All timestamps are non-decreasing
* $1 \leq \text{timestamp} \leq 2 \times 10^9$
* At most $300$ calls per test (per problem constraints)


## Solutions

### C#

In [ ]:
public class HitCounter {
    private int[] times = new int[300];   // last timestamp that wrote to each slot
    private int[] hits  = new int[300];   // hit count for that timestamp

    public void Hit(int timestamp) {
        int slot = timestamp % 300;
        if (times[slot] != timestamp) {   // slot belongs to an older second — reset it
            times[slot] = timestamp;
            hits[slot] = 0;
        }
        hits[slot]++;
    }

    public int GetHits(int timestamp) {
        int total = 0;
        for (int i = 0; i < 300; i++)
            // Only count slots that recorded a timestamp within the last 300 seconds
            if (timestamp - times[i] < 300)
                total += hits[i];
        return total;
    }
}

### Python

In [ ]:
class HitCounter:
    def __init__(self):
        # 300-slot circular buffer; each slot stores (last_timestamp, count)
        self.times = [0] * 300
        self.hits  = [0] * 300

    def hit(self, timestamp: int) -> None:
        slot = timestamp % 300
        if self.times[slot] != timestamp:
            # Slot was written by an earlier second — overwrite it
            self.times[slot] = timestamp
            self.hits[slot]  = 0
        self.hits[slot] += 1

    def getHits(self, timestamp: int) -> int:
        # Sum slots that fall within the [timestamp-299, timestamp] window
        return sum(
            self.hits[i]
            for i in range(300)
            if timestamp - self.times[i] < 300
        )

### Go

In [ ]:
type HitCounter struct {
    times [300]int
    hits  [300]int
}

func Constructor() HitCounter { return HitCounter{} }

func (h *HitCounter) Hit(timestamp int) {
    slot := timestamp % 300
    if h.times[slot] != timestamp {
        // Slot is stale — belongs to a second ≥ 300 earlier; reset it
        h.times[slot] = timestamp
        h.hits[slot]  = 0
    }
    h.hits[slot]++
}

func (h *HitCounter) GetHits(timestamp int) int {
    total := 0
    for i := 0; i < 300; i++ {
        // Only include slots whose timestamp is within the last 300 seconds
        if timestamp-h.times[i] < 300 {
            total += h.hits[i]
        }
    }
    return total
}

### Rust

In [ ]:
struct HitCounter {
    times: [i32; 300],
    hits:  [i32; 300],
}

impl HitCounter {
    fn new() -> Self {
        HitCounter { times: [0; 300], hits: [0; 300] }
    }

    fn hit(&mut self, timestamp: i32) {
        let slot = (timestamp % 300) as usize;
        if self.times[slot] != timestamp {
            // Slot is stale (written ≥ 300 s ago) — clear and reuse
            self.times[slot] = timestamp;
            self.hits[slot]  = 0;
        }
        self.hits[slot] += 1;
    }

    fn get_hits(&self, timestamp: i32) -> i32 {
        // Accumulate counts for all slots within the last 300 seconds
        (0..300)
            .filter(|&i| timestamp - self.times[i] < 300)
            .map(|i| self.hits[i])
            .sum()
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `hit(1)`, `hit(2)`, `hit(3)`, `getHits(4)` → **3**
Slots 1%300, 2%300, 3%300 each hold count 1. At timestamp 4, all three are within [4-299, 4], so all are counted.

### 2. Slightly Complex
**Input:** `hit(1)` ×300, `getHits(300)` → **300**, `getHits(301)` → **0**
Slot `1 % 300 = 1` accumulates count 300 at timestamp 1. At `getHits(300)`, `300 - 1 = 299 < 300` ✓. At `getHits(301)`, `301 - 1 = 300` which is not $< 300$, so the slot is excluded → **0**.

### 3. Edge Case: Time Factor
**Input:** `hit(t)` for $t = 1, 2, \ldots, 300$, then `getHits(300)`
All 300 slots are populated (one per second). `getHits` iterates all 300 slots in exactly 300 steps — confirming the $O(300) = O(1)$ bound and returning **300**.

### 4. Edge Case: Space Factor
**Input:** $10^9$ consecutive distinct timestamps, one hit each
The circular buffer always has at most 300 live slots. Regardless of how many hits arrive, the fixed 600-integer array (`times` + `hits`) is the only allocation — $O(1)$ space.

### 5. Almost-Impossible but Plausible
**Input:** `hit(300)`, `hit(600)` (same slot: both map to slot 0), `getHits(600)` → **1**
Timestamp 600 overwrites slot 0 (was timestamp 300, now stale: $600 - 300 = 300 \not< 300$). After overwrite, slot 0 holds `(600, 1)`. `getHits(600)` finds `600 - 600 = 0 < 300` ✓ → returns **1**. The stale overwrite is the core correctness guarantee of the circular buffer design.
